# 05 — Save and Load

**Purpose:** Show the full round-trip: save a `Trace` to a portable `.tlspec` bundle,
inspect what lands on disk, load it back, and verify field survival.  Also covers
save levels, `PayloadLoadHints`, and lazy loading.

**Surfaces covered:**
- [ ] `tl.save(trace, path)` — basic save (level='portable')
- [ ] `tl.save` kwarg: `level` ('portable' | 'audit' | 'executable_with_callables')
- [ ] `tl.save` kwarg: `include_outs`, `include_grads`, `include_saved_args`, `include_rng_states`
- [ ] `.tlspec` bundle layout (directory tree on disk)
- [ ] `manifest.json` — schema, blob index, model fingerprint
- [ ] `tl.load(path)` — load and restore a `Trace`
- [ ] Round-trip field survival (activation tensors, layer labels, metadata)
- [ ] `tl.load` kwarg: `lazy`, `map_location`
- [ ] `PayloadLoadHints` — class inspection
- [ ] `tl.load` return type when loading a bundle (Trace vs Bundle)

## 1. Environment setup

In [ ]:
import pathlib
import sys
import json
import os
import tempfile

sys.path.insert(0, str(pathlib.Path.cwd()))

import torch
import torchlens as tl
from _models import ZOO

print(f"torchlens version : {tl.__version__}")
print(f"torch version     : {torch.__version__}")

# All disk writes go under a single temp dir; cleaned up at process exit
TMPDIR = tempfile.mkdtemp(prefix="tl_audit_05_")
print(f"temp dir          : {TMPDIR}")

## 2. Capture a trace

In [ ]:
model, x = ZOO["tiny_mlp"]()
trace = tl.trace(model, x)

print(repr(trace))
print()
print("layer_labels:", trace.layer_labels)
print("relu activation value (original):")
relu_orig = trace["relu_1_2"].out.clone()
print(" ", relu_orig)

## 3. `tl.save` — write to disk

In [ ]:
import inspect

print("tl.save signature:")
print(inspect.signature(tl.save))
print()
print("Levels available: 'portable' (default), 'audit', 'executable_with_callables'")

In [ ]:
# Save with default level='portable'
save_path = os.path.join(TMPDIR, "trace_portable.tlspec")

tl.save(trace, save_path)
print(f"Saved to: {save_path}")
print(f"Is directory: {os.path.isdir(save_path)}")

## 4. `.tlspec` bundle layout — what lands on disk

In [ ]:
def show_bundle_tree(bundle_path: str) -> None:
    """Print the directory tree and file sizes of a .tlspec bundle."""
    root = pathlib.Path(bundle_path)
    print(f"{root.name}/")
    for item in sorted(root.rglob("*")):
        rel = item.relative_to(root)
        depth = len(rel.parts) - 1
        indent = "    " * depth + "  "
        if item.is_dir():
            print(f"{indent}{item.name}/")
        else:
            size = item.stat().st_size
            print(f"{indent}{item.name}  ({size:,} B)")


show_bundle_tree(save_path)

## 5. `manifest.json` — human-readable bundle header

In [ ]:
manifest_path = os.path.join(save_path, "manifest.json")
with open(manifest_path) as f:
    manifest = json.load(f)

# Show key top-level fields
top_level_keys = [
    "kind",
    "tlspec_version",
    "schema_version",
    "save_level",
    "torchlens_version",
    "torch_version",
    "python_version",
    "n_layers",
    "n_out_blobs",
    "n_grad_blobs",
    "body_format",
    "created_at",
    "model_signature",
]
print("=== manifest.json top-level fields ===")
for k in top_level_keys:
    if k in manifest:
        print(f"  {k}: {manifest[k]}")

print()
print("=== model_fingerprint ===")
for k, v in manifest.get("model_fingerprint", {}).items():
    if k != "extra_metadata":
        print(f"  {k}: {v}")

print()
print("=== blob index (first 2 entries) ===")
for entry in manifest.get("body_index", [])[:2]:
    # sha256 is the content-addressed blob hash; truncate for display (full 64-char
    # hex is noisy and trips entropy-based secret scanners on a committed notebook).
    shown = dict(entry)
    if "sha256" in shown:
        shown["sha256"] = shown["sha256"][:12] + "..."
    print(f"  {shown}")

print()
print("=== sites (layer->function mapping) ===")
for site in manifest.get("sites", []):
    print(f"  {site['layer_label']:15s}  func={site['function_path']}")

## 6. `tl.load` — restore from disk

In [ ]:
print("tl.load signature:")
print(inspect.signature(tl.load))

In [ ]:
loaded_trace = tl.load(save_path)

print("loaded type:", type(loaded_trace).__name__)
print("loaded repr:", repr(loaded_trace))
print()
print("layer_labels (loaded):", loaded_trace.layer_labels)

## 7. Round-trip field survival — verify activations and metadata

In [ ]:
# Check relu activation tensor survives
relu_loaded = loaded_trace["relu_1_2"].out

print("Original relu.out:")
print(" shape:", relu_orig.shape, " dtype:", relu_orig.dtype)
print(" values:", relu_orig)
print()
print("Loaded relu.out:")
print(" shape:", relu_loaded.shape, " dtype:", relu_loaded.dtype)
print(" values:", relu_loaded)
print()
print("torch.allclose match:", torch.allclose(relu_orig, relu_loaded))

In [ ]:
# Check a few metadata fields survive
orig_relu_op = trace["relu_1_2"]
loaded_relu_op = loaded_trace["relu_1_2"]

fields_to_check = ["layer_label", "layer_type", "activation_memory"]
print("Metadata field survival:")
print(f"  {'field':25s}  {'original':25s}  {'loaded':25s}  match")
print("  " + "-" * 85)
for field in fields_to_check:
    try:
        orig_val = getattr(orig_relu_op, field, "<missing>")
        load_val = getattr(loaded_relu_op, field, "<missing>")
        match = orig_val == load_val
        print(f"  {field:25s}  {str(orig_val):25s}  {str(load_val):25s}  {match}")
    except Exception as e:
        print(f"  {field:25s}  ERROR: {e}")

## 8. Save levels — `'portable'` vs `'audit'` vs `'executable_with_callables'`

In [ ]:
# Save at all three levels and compare bundle sizes
levels = ["portable", "audit", "executable_with_callables"]
sizes = {}

for level in levels:
    lpath = os.path.join(TMPDIR, f"trace_{level}.tlspec")
    try:
        tl.save(trace, lpath, level=level)
        # Compute total bundle size
        total = sum(f.stat().st_size for f in pathlib.Path(lpath).rglob("*") if f.is_file())
        # Read save_level from manifest to confirm
        with open(os.path.join(lpath, "manifest.json")) as fp:
            mfst = json.load(fp)
        saved_level = mfst.get("save_level", "?")
        sizes[level] = total
        print(f"  level='{level}' -> manifest save_level='{saved_level}'  bundle size: {total:,} B")
    except Exception as e:
        print(f"  level='{level}' -> ERROR: {type(e).__name__}: {e}")

## 9. Selective save — omitting grads or outs

In [ ]:
# Save without output tensors (metadata-only bundle)
meta_path = os.path.join(TMPDIR, "trace_no_outs.tlspec")
tl.save(trace, meta_path, include_outs=False)

with open(os.path.join(meta_path, "manifest.json")) as f:
    meta_manifest = json.load(f)

print("include_outs=False ->")
print(f"  n_out_blobs: {meta_manifest['n_out_blobs']}  (expected 0)")
print(f"  body_index entries: {len(meta_manifest['body_index'])}")

# Load and try to access an activation tensor
meta_loaded = tl.load(meta_path)
try:
    _ = meta_loaded["relu_1_2"].out
    print("  WARNING: out available unexpectedly")
except Exception as e:
    print(f"  Accessing .out after include_outs=False raises: {type(e).__name__}")

## 10. `tl.load` with `lazy=True` and `map_location`

In [ ]:
# Lazy load: out blobs remain None until explicitly materialized.
# Use lazy=True when you only need metadata and want to selectively load tensors.
lazy_trace = tl.load(save_path, lazy=True)
print("lazy load repr:", repr(lazy_trace))
print()

# With lazy=True, .out is None until materialize_out() is called
lazy_op = lazy_trace["relu_1_2"]
print("lazy_op.out (before materialize):", lazy_op.out)  # None
print()

# Materialize a specific tensor on demand
lazy_relu = lazy_op.materialize_out()
print("After materialize_out():")
print(" shape:", lazy_relu.shape)
print(" matches original:", torch.allclose(relu_orig, lazy_relu))

In [ ]:
# map_location — force tensors to a specific device on load
cpu_trace = tl.load(save_path, map_location="cpu")
print("map_location='cpu' load:")
relu_device = cpu_trace["relu_1_2"].out.device
print(" relu.out.device:", relu_device)

## 11. `PayloadLoadHints` — class inspection

In [ ]:
print("PayloadLoadHints class:")
print(" module:", tl.PayloadLoadHints.__module__)
print(" signature:", inspect.signature(tl.PayloadLoadHints.__init__))
print()
print(" Parameters:")
print("   jax: JaxPayloadLoadHint | None  -- JAX-specific tensor loading options")
print()
print("JaxPayloadLoadHint class:")
print(" signature:", inspect.signature(tl.JaxPayloadLoadHint.__init__))
print()

# Construct a default PayloadLoadHints (torch-only; no JAX hints needed)
hints = tl.PayloadLoadHints()
print("PayloadLoadHints() instance:", hints)
print(" hints.jax:", hints.jax)

In [ ]:
# Load with explicit PayloadLoadHints (no-op for torch-only bundles)
hinted_trace = tl.load(save_path, payload_hints=tl.PayloadLoadHints())
print("load with payload_hints:")
print(" type:", type(hinted_trace).__name__)
print(" repr:", repr(hinted_trace))

## 12. Saving a more complex model (batch_norm — with buffer layers)\n\nDemonstrates a model with buffers (running_mean, running_var), which adds more blobs.

In [ ]:
# Use a model without non-picklable backward hooks (demo_model has torch.rand which
# creates ViewBackward0 hooks that cannot be pickled; use batch_norm instead)
bn_model, bn_x = ZOO["batch_norm"]()
bn_trace = tl.trace(bn_model, bn_x)
print("batch_norm trace:", repr(bn_trace))
print("layers:", bn_trace.layer_labels)

bn_path = os.path.join(TMPDIR, "bn_trace.tlspec")
tl.save(bn_trace, bn_path)

# Show bundle structure (more layers = more blobs)
show_bundle_tree(bn_path)

# Round-trip
bn_loaded = tl.load(bn_path)
print()
print("loaded layer_labels:", bn_loaded.layer_labels)
print("layers match:", bn_trace.layer_labels == bn_loaded.layer_labels)

---

## ⚠️ GAPs / ergonomic smells

- **`.tlspec` is a directory, not a file.** Users passing `path.tlspec` get a directory;
  standard OS copy (`cp file.tlspec /dest/`) silently requires `-r`.  This is a potential
  UX landmine for users who expect `.tlspec` to be a ZIP or a flat file.

- **`lazy=True` sets `.out = None`; user must call `op.materialize_out()` explicitly.**
  The docstring documents this correctly, but the pattern is non-obvious: indexing the
  lazy trace still returns an Op object, it just has `None` payloads.  The discoverability
  path (`op.materialize_out()`) is not shown in the repr or error message.

- **`include_outs=False` then `.out` access raises an exception** but the error message
  could be more explicit that the payload was excluded at save time vs corrupted.

- **`PayloadLoadHints` is only meaningful for JAX bundles.** For pure PyTorch use, the
  constructor has no useful parameters.  The class is in `__all__` but is essentially a
  no-op for the common case — a candidate for removal from the public surface or for a
  more future-proof design.

- **`tl.load` return type annotation says `Trace | Bundle | InterventionSpec`** but the
  docstring/example only shows `Trace`.  Users loading a bundle need to know to check
  `isinstance(result, tl.Bundle)`.

- No GAPs in the core `save`/`load` round-trip — all tensor values and layer labels
  survive correctly.